This notebook builds the Gold layer by aggregating the cleansed Silver sales data into business level metrics suitable for reporting and analysis. Monthly aggregations are computed to derive total revenue, total units sold, and total number of orders, providing a concise and query optimized view of sales performance over time. The Gold dataset represents the final consumption layer of the data lake, designed for direct use by BI tools and analytical workloads.

In [ ]:
# Import the necessary libraries 
from pyspark.sql import SparkSession
from pathlib import Path
import src.sqlqueries as sq
import sys
import os
import warnings
import utils.logger as logger
from pyspark.sql import functions as F
from pyspark.sql.functions import to_timestamp, col
from pyspark.sql.functions import sum as spark_sum, when
warnings.filterwarnings("ignore")


#Set the path for logging outputs
job_name = "sales_kpis"
data_base_path = Path("../Logs") # path for logging data
data_working_path = os.path.join(data_base_path, job_name) 
os.makedirs(data_working_path, exist_ok=True)
logger.set_logging_path(data_working_path)

# Spark initialization locally for development
spark = (
    SparkSession.builder
    .appName("sales-bronze-ingestion")
    .getOrCreate()
)

logger.log("Spark Session initialized")

df = spark.read.parquet(
    "../data/cleansed/sales"
)
logger.log("Spark DataFrame created from silver - cleansed sales parquet files")

**Gold Aggregations**

In [ ]:
gold_df = (
    df.groupBy("Order_Year", "Order_Month")
      .agg(
          F.sum(F.col("Quantity_Ordered") * F.col("Price_Each")).alias("Total_Revenue"),
          F.sum("Quantity_Ordered").alias("Total_Units"),
          F.countDistinct("Order_ID").alias("Total_Orders")
      )
)


In [3]:
gold_df.write.mode("overwrite").parquet("../data/gold/sales_monthly")
